# 06 - Pair Export and Analysis

Assembles the prediction log into per-scene comparisons and reports the outcome
of each comparison. The analysis uses on the constructed set produced in
Notebook 04. Every same-side scene is paired with its opposite configuration,
built from the same base frame, enabling a decisive within-frame contrast.

Results are reported in an instrument-first order. If the lateral output does not
respond to the stimulus, or if the measured effect sits below the quantisation
floor, downstream directional claims cannot be interpreted. In that case, a
language result could reflect an instrument failure rather than spatial
grounding. The contrastive experiments are reported next, followed by the control
conditions.

Statistics are computed on the continuous expected-bin readout rather than on the
quantised argmax action, because the quantisation step is comparable to the
measured effect size. Coverage is reported rather than assumed.

Each comparison is evaluated on the axis targeted by its spatial term, using
`axis_index`. 

Null results are tested with an equivalence test against one action-bin width, not
with a non-significant p value. 

## 1. Mount Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')
import os
CACHE_DIR = '/content/drive/MyDrive/openvla_cache/v2/bridge'
CONSTRUCTED_DIR = '/content/drive/MyDrive/openvla_cache/v2/constructed'
PROBE_CSV = '/content/drive/MyDrive/openvla_cache/v2/probe_predictions.csv'
PAIRS_JSON = '/content/drive/MyDrive/openvla_cache/v2/pairs.json'
MANIFEST = os.path.join(CACHE_DIR, 'manifest.csv')
print('log      ->', PROBE_CSV)
print('manifest ->', MANIFEST)

Mounted at /content/drive
log      -> /content/drive/MyDrive/openvla_cache/v2/probe_predictions.csv
manifest -> /content/drive/MyDrive/openvla_cache/v2/bridge/manifest.csv


## 2. Import the code

Clones the project code from GitHub into the runtime, so the analysis functions
always match the pushed commit. No OpenVLA load is needed.

In [2]:
import sys, os, importlib, subprocess

REPO_URL = 'https://github.com/LewisTL/ECS8056.git'
BRANCH = 'master'
REPO_DIR = '/content/ECS8056'

def sync_repo():
    """Clone or hard-refresh the repository so it matches origin/BRANCH."""
    token = os.environ.get('GITHUB_TOKEN', '')
    url = REPO_URL.replace('https://', f'https://{token}@') if token else REPO_URL
    if os.path.isdir(os.path.join(REPO_DIR, '.git')):
        subprocess.run(['git', '-C', REPO_DIR, 'remote', 'set-url', 'origin', url],
                       check=True)
        subprocess.run(['git', '-C', REPO_DIR, 'fetch', '--quiet', '--depth', '1',
                        'origin', BRANCH], check=True)
        subprocess.run(['git', '-C', REPO_DIR, 'reset', '--hard', '--quiet',
                        f'origin/{BRANCH}'], check=True)
    else:
        subprocess.run(['git', 'clone', '--quiet', '--depth', '1', '--branch',
                        BRANCH, url, REPO_DIR], check=True)
    return subprocess.run(['git', '-C', REPO_DIR, 'rev-parse', '--short', 'HEAD'],
                          capture_output=True, text=True).stdout.strip()


commit = sync_repo()
module_dir = REPO_DIR
if module_dir not in sys.path:
    sys.path.insert(0, module_dir)
for _m in ('action_bins', 'prediction_log', 'data', 'controls', 'compose_scenes', 'export_pairs', 'analysis',):
    sys.modules.pop(_m, None)
importlib.invalidate_caches()

import analysis
from analysis import (axis_value, wilcoxon_paired, tost_equivalence,
                      resolution_report, mirror_check,
                      mirror_check_by_configuration, lexical_check,
                      term_effect, congruence_test, absolute_congruence,
                      same_side_test, CONTINUOUS_COLS)
from export_pairs import load_inputs, build_pairs, write_pairs
from compose_scenes import (evaluation_scenes, load_constructed_manifest,
                            manipulation_rate, IMAGE_X_TO_LATERAL_SIGN)
from data import AXIS_INDEX
print(f'imported project modules from {module_dir} @ {commit}')

imported project modules from /content/ECS8056 @ 050bcc0


## 3. Load the log and report coverage

Every result below is conditional on what was collected, so the log composition
is printed before any statistic. A condition is skipped when its precondition fails.

In [3]:
import numpy as np
import pandas as pd

log = pd.read_csv(PROBE_CSV)
n_bridge = int((log['scene_source'] == 'bridge').sum())
print(f'{len(log)} predictions in the log')
print(f'{n_bridge} unaltered Bridge rows held back')

coverage = log[CONTINUOUS_COLS].notna().all(axis=1)
constructed = log['scene_source'] == 'constructed'
usable = log[coverage & constructed].copy()
print(f'\ncontinuous readout present on {coverage.sum()}/{len(log)} rows '
      f'({coverage.mean():.1%})')
if coverage.mean() < 1.0:
    print('rows without it are excluded from the continuous comparisons')

usable['value'] = axis_value(usable)
print(f'\nconstructed predictions used: {len(usable)}')
print('\nby condition:')
print(usable['condition'].value_counts().sort_index().to_string())
print('\nby axis:', dict(usable['axis'].value_counts()))

2720 predictions in the log
0 unaltered Bridge rows held back

continuous readout present on 2720/2720 rows (100.0%)

constructed predictions used: 2720

by condition:
condition
baseline          680
mirror            680
mirror_neutral    340
neutral           340
swapped_scene     680

by axis: {'lateral': np.int64(2720)}


## 4. Export pairs

Groups the constructed rows of the log into comparisons by
(pair_id, frame, condition, scene_source) and writes `pairs.json` for the Isaac
Sim visualiser. Unaltered Bridge predictions are held back.

Scene labels are re-joined from the manifest rather than read from the log. The
log freezes whatever a label was when the probe ran, which previously hid every
label added by review afterwards.

In [4]:
preds, manifest = load_inputs(PROBE_CSV, MANIFEST)
n_bridge_preds = int((preds['scene_source'] == 'bridge').sum())
preds = preds[preds['scene_source'] == 'constructed']
print(f'held back {n_bridge_preds} unaltered Bridge predictions from the export')
pairs, stats = build_pairs(preds, manifest)
print(f"{stats['paired']} paired comparisons, {stats['neutral']} single-role records")
if stats['incomplete']:
    print(f"{stats['incomplete']} groups are missing a role the condition calls "
          "for, the probe run was most likely interrupted")
write_pairs(pairs, PAIRS_JSON)

/content/drive/MyDrive/openvla_cache/v2/probe_predictions.csv
  67 columns, 2720 rows, consistent
held back 0 unaltered Bridge predictions from the export
1020 paired comparisons, 680 single-role records
[write_pairs] wrote 1700 pairs -> /content/drive/MyDrive/openvla_cache/v2/pairs.json


'/content/drive/MyDrive/openvla_cache/v2/pairs.json'

## 5. Instrument check: is the lateral channel live?

With the instruction held fixed, mirroring the image reverses the lateral axis of
the scene, so a model that reads lateral position must change the sign of its
lateral output. The term-stripped variant isolates object grounding from any
influence of the spatial word.

If the response does not move when the whole scene is
reflected, then the visual channel is not live on this axis, and a null on the
language comparisons follows from the model ignoring the image rather than from
anything about spatial language. 

Passing this check is a necessary condition for the language claims, not evidence
for them. Reaching toward the only object in view requires no spatial language at
all.

The constructed scenes are reported per arrangement rather than pooled, because
the arrangements are not interchangeable here. Reflecting an `opposite`
arrangement maps the layout onto itself, so a response near the midpoint produces
an antisymmetry and an invariance both near zero. The
same-side arrangements are where a reflection genuinely moves the scene. Pooling
them would dilute the only stratum that can pass or fail the check.

In [5]:
def print_mirror(result, label):
    print(f'--- {label}')
    for kind in ('neutral', 'term'):
        entry = result.get(kind, {})
        if not entry.get('n'):
            print(f'  {kind}: no paired mirror predictions')
            continue
        print(f"  {kind:8} n={entry['n']:4}  flips={entry['flip_rate']:.1%}  "
              f"identical={entry['identical_rate']:.1%}  "
              f"mean |lateral|={entry['mean_abs_original']:.5f}  "
              f"mean |change|={entry['mean_abs_change']:.5f}")
        print(f"           antisymmetry median={entry['antisymmetry']['median']:+.5f} "
              f"p={entry['antisymmetry']['p_value']:.3g}   "
              f"invariance median={entry['invariance']['median']:+.5f} "
              f"p={entry['invariance']['p_value']:.3g}")
    return result

def print_mirror(result, label):
    print(f'--- {label}')
    for kind in ('neutral', 'term'):
        entry = result.get(kind, {})
        if not entry.get('n'):
            print(f'  {kind}: no paired mirror predictions')
            continue
        print(f"  {kind:8} n={entry['n']:4}  flips={entry['flip_rate']:.1%}  "
              f"identical={entry['identical_rate']:.1%}  "
              f"mean |lateral|={entry['mean_abs_original']:.5f}  "
              f"mean |change|={entry['mean_abs_change']:.5f}")
        print(f"           antisymmetry median={entry['antisymmetry']['median']:+.5f} "
              f"p={entry['antisymmetry']['p_value']:.3g}   "
              f"invariance median={entry['invariance']['median']:+.5f} "
              f"p={entry['invariance']['p_value']:.3g}")
    return result

lateral = usable[usable['axis_index'] == AXIS_INDEX['lateral']]

mirror_built = mirror_check_by_configuration(lateral)
for configuration, result in sorted(mirror_built.items()):
    print_mirror(result, configuration or 'unlabelled')

mirror_built_same_side = mirror_check(
    lateral[lateral['configuration'].astype(str).str.startswith('same_side')])

mirror_built_same_side = mirror_check(
    lateral[(lateral['scene_source'] == 'constructed')
            & lateral['configuration'].astype(str).str.startswith('same_side')])

--- opposite
  neutral  n= 151  flips=49.0%  identical=0.0%  mean |lateral|=0.00349  mean |change|=0.00700
           antisymmetry median=-0.00118 p=0.000247   invariance median=+0.00046 p=0.288
  term     n= 151  flips=41.7%  identical=0.0%  mean |lateral|=0.00362  mean |change|=0.00716
           antisymmetry median=-0.00182 p=4.08e-06   invariance median=+0.00010 p=0.737
--- same_side_left
  neutral  n=  93  flips=61.3%  identical=0.0%  mean |lateral|=0.00357  mean |change|=0.00807
           antisymmetry median=-0.00108 p=0.0346   invariance median=+0.00219 p=2.1e-08
  term     n=  93  flips=54.8%  identical=0.0%  mean |lateral|=0.00380  mean |change|=0.00832
           antisymmetry median=-0.00155 p=5.79e-05   invariance median=+0.00190 p=1.88e-08
--- same_side_right
  neutral  n=  96  flips=54.2%  identical=0.0%  mean |lateral|=0.00584  mean |change|=0.00987
           antisymmetry median=-0.00175 p=0.000457   invariance median=-0.00272 p=8.81e-13
  term     n=  96  flips=55.2%  

## 6. Measurement resolution

A distribution dominated by exact zeros, or by differences smaller than one bin, is at the
resolution floor of the argmax readout, and no conclusion drawn from its sign is
safe.

This is reported because it is
the defect that made the earlier findings uninterpretable. The median paired
difference was exactly zero in every stratum, and five of eight sampled pairs
were bit-identical between the two instructions. The comparison between the
argmax and continuous columns here is the direct evidence of what the continuous
readout recovered.

In [6]:
BIN_WIDTH = 0.000325 # Set from the model's own decoding constants, printed by Notebooks 03 and 05.

def paired_difference(frame, condition='baseline', continuous=True):
    """Signed A minus B difference per scene on that scene's own axis."""
    sub = frame[frame['condition'] == condition].copy()
    if sub.empty:
        return pd.DataFrame(columns=['scene_id', 'diff'])
    sub['value'] = axis_value(sub, continuous=continuous)
    wide = sub.pivot_table(index='scene_id', columns='role', values='value')
    if not {'a', 'b'} <= set(wide.columns):
        return pd.DataFrame(columns=['scene_id', 'diff'])
    wide = wide.dropna(subset=['a', 'b'])
    return pd.DataFrame({'scene_id': wide.index, 'diff': (wide['a'] - wide['b']).to_numpy()})

rows = []
for source in sorted(usable['scene_source'].unique()):
    frame = usable[usable['scene_source'] == source]
    for readout, flag in (('argmax', False), ('continuous', True)):
        diffs = paired_difference(frame, continuous=flag)['diff']
        report = resolution_report(diffs, BIN_WIDTH)
        rows.append({'scene_source': source, 'readout': readout, **report})

resolution = pd.DataFrame(rows)
print(resolution[['scene_source', 'readout', 'n', 'frac_exact_zero',
                  'frac_below_bin', 'n_distinct']].to_string(index=False))
print(f'\nbin width used: {BIN_WIDTH:.7f}')


scene_source    readout   n  frac_exact_zero  frac_below_bin  n_distinct
 constructed     argmax 340         0.326471        0.444118         175
 constructed continuous 340         0.000000        0.361765         330

bin width used: 0.0003250


## 7. The experiments: constructed scenes

### 7a. Direction of the paired difference

This is a direction check rather than the decisive test. In the `opposite` configuration the term's
conventional direction and the actual layout coincide by construction, so a model
that maps `left` to a fixed direction without ever consulting the scene agrees
here just as completely as a grounded one. What the statistic establishes is that
the difference is oriented consistently with the geometry at all, which is a
precondition for reading its sign, and it fixes the convention relating image
position to the lateral action empirically. Sections 7b and 7c carry the actual
discrimination.

In [7]:
built = usable[usable['scene_source'] == 'constructed']
if built.empty:
    print('no constructed predictions in the log, run Notebooks 04 and 05 first')
else:
    congruence = congruence_test(built, lateral_sign=IMAGE_X_TO_LATERAL_SIGN)
    print(f"n={congruence['n']}  agreement with geometry={congruence['agreement']:.1%}")
    test = congruence['test']
    print(f"  median oriented difference={test['median']:+.6f}  "
          f"p={test['p_value']:.3g}  effect size={test['rank_biserial']:+.2f}")
    print('\nby configuration:')
    for name, entry in sorted(congruence['by_configuration'].items()):
        print(f"  {name:16} n={entry['n']:4}  agreement={entry['agreement']:.1%}  "
              f"p={entry['test']['p_value']:.3g}  "
              f"effect={entry['test']['rank_biserial']:+.2f}")
    print(f'\nsign convention in use: IMAGE_X_TO_LATERAL_SIGN = '
          f'{IMAGE_X_TO_LATERAL_SIGN:+d}')

n=340  agreement with geometry=46.8%
  median oriented difference=-0.000006  p=0.485  effect size=-0.04

by configuration:
  opposite         n= 151  agreement=47.7%  p=0.529  effect=-0.06
  same_side_left   n=  93  agreement=47.3%  p=0.829  effect=-0.03
  same_side_right  n=  96  agreement=44.8%  p=0.869  effect=-0.02

sign convention in use: IMAGE_X_TO_LATERAL_SIGN = -1


### 7b. The decisive comparison: same side versus opposite

When both instances lie on the same side of the start position, scene grounding
and a word-to-direction mapping make opposite predictions. A grounded model
selects between two targets that share a direction, so both instructions produce
same-signed actions that differ only in magnitude. A mapping from `left` to one
direction and `right` to the other produces opposite signs, because the
arrangement does not enter the mapping.

On the `opposite` configuration the two accounts coincide. The original
sign-flip metric scored that shared behaviour, so a high rate was consistent
with either account. The contrast between configurations is what separates them,
and it is available only because the same-side arrangement is constructed.

The contrast is reported in two forms that follow from how the set is built.
It is computed paired within the base frame as well as across all scenes. The
frozen set constructs both arrangements from one frame, so the paired comparison
holds background, object, cutout, and instruction fixed and leaves arrangement
as the remaining difference. It is also computed twice, once over every nonzero
prediction and once over predictions of at least one action bin. A sub-bin sign
is not an executable decision, treating it as one pulls every rate toward one
half.

In [8]:
if not built.empty:
    for label, floor in (('every nonzero prediction', 0.0),
                         (f'at least one bin ({BIN_WIDTH:.6f})', BIN_WIDTH)):
        same_side = same_side_test(built, min_magnitude=floor)
        print(f'--- same-signed action rate, {label}')
        for name, entry in sorted(same_side['by_configuration'].items()):
            print(f"  {name:16} n={entry['n']:4}  "
                  f"same sign={entry['same_sign_rate']:.1%}  "
                  f"undecided={entry['n_unresolved']}  "
                  f"median |magnitude gap|={entry['median_magnitude_gap']:.6f}")
        contrast = same_side['contrast']
        print(f"  contrast (same side minus opposite): "
              f"{contrast['difference']:+.1%}  p={contrast['p_value']:.3g}")
        paired = same_side['contrast_paired']
        print(f"  paired within the base frame: {paired['difference']:+.1%}  "
              f"pairs={paired['n_pairs']}  discordant={paired['n_discordant']}  "
              f"p={paired['p_value']:.3g}  unpaired scenes={paired['n_unpaired']}")

--- same-signed action rate, every nonzero prediction
  opposite         n= 151  same sign=81.5%  undecided=0  median |magnitude gap|=0.000536
  same_side_left   n=  93  same sign=76.3%  undecided=0  median |magnitude gap|=0.000629
  same_side_right  n=  96  same sign=91.7%  undecided=0  median |magnitude gap|=0.000867
  contrast (same side minus opposite): +2.7%  p=0.563
  paired within the base frame: +0.0%  pairs=128  discordant=32  p=1  unpaired scenes=63
--- same-signed action rate, at least one bin (0.000325)
  opposite         n= 117  same sign=83.8%  undecided=34  median |magnitude gap|=0.000609
  same_side_left   n=  56  same sign=83.9%  undecided=37  median |magnitude gap|=0.000807
  same_side_right  n=  86  same sign=93.0%  undecided=10  median |magnitude gap|=0.000950
  contrast (same side minus opposite): +5.7%  p=0.199
  paired within the base frame: +4.1%  pairs=74  discordant=11  p=0.549  unpaired scenes=94


### 7c. Does each instruction move toward its own target?

The same discrimination, stated at the instruction rather than the pair. Each
instruction is scored against the side its own target occupies, not against the
relative order of the two targets.

On a same-side scene both targets share a direction. A grounded model therefore
agrees on both instructions. A word-to-direction mapping sends `left` one way
and `right` the other, so only one of those actions can point at a target.
Expected agreement is near one under grounding and near one half under a
lexical mapping. The paired difference does not separate the accounts, both
produce the same difference on these scenes.

The instruction-level view is also pooled by congruency, whether an instruction's
term names the side its target occupies. Both accounts agree on the congruent
instruction. They differ on the incongruent one. Pooling across arrangements
doubles the sample for that contrast, because each same-side scene contributes
one incongruent instruction whichever side it was built on.

The comparison is repeated on the mirrored stimuli. Reflection negates every
recorded side, so the same quantity is measured on images the model has not
been shown, using predictions the control factorial already collects.

In [9]:
def report_absolute(result, label):
    if not result.get('n'):
        print(f'--- {label}: no resolved predictions')
        return
    print(f"--- {label}")
    print(f"  n={result['n']}  ({result['n_unresolved']} scenes gave no decidable "
          "prediction and pick no side)")
    print(f"  instructions moving toward their own target: "
          f"{result['agreement']:.1%}")
    print(f"  both instructions correct in the same scene: "
          f"{result['both_correct']:.1%}")
    print('  by configuration:')
    for name, entry in sorted(result['by_configuration'].items()):
        if not entry.get('n'):
            print(f'    {name:16} no resolved predictions')
            continue
        print(f"    {name:16} n={entry['n']:4}  agreement={entry['agreement']:.1%}  "
              f"both correct={entry['both_correct']:.1%}  "
              f"(A={entry['agreement_a']:.1%}, B={entry['agreement_b']:.1%})")
    print('  by congruency of the instruction with its own target:')
    for name, entry in sorted(result.get('by_congruency', {}).items()):
        if not entry.get('n'):
            continue
        print(f"    {name:12} n={entry['n']:4}  agreement={entry['agreement']:.1%}")


if not built.empty:
    absolute = absolute_congruence(built, lateral_sign=IMAGE_X_TO_LATERAL_SIGN)
    if not absolute.get('n'):
        print('target sides were not logged, re-run Notebook 05 so the '
              'constructed scenes carry the recorded target sides')
    else:
        report_absolute(absolute, 'every nonzero prediction')
        report_absolute(
            absolute_congruence(built, lateral_sign=IMAGE_X_TO_LATERAL_SIGN,
                                min_magnitude=BIN_WIDTH),
            f'at least one bin ({BIN_WIDTH:.6f})')
        report_absolute(
            absolute_congruence(built, lateral_sign=IMAGE_X_TO_LATERAL_SIGN,
                                condition='mirror', geometry_sign=-1),
            'replication on the mirrored stimuli')

--- every nonzero prediction
  n=340  (0 scenes gave no decidable prediction and pick no side)
  instructions moving toward their own target: 63.4%
  both instructions correct in the same scene: 40.9%
  by configuration:
    opposite         n= 151  agreement=48.0%  both correct=7.3%  (A=31.1%, B=64.9%)
    same_side_left   n=  93  agreement=58.1%  both correct=46.2%  (A=55.9%, B=60.2%)
    same_side_right  n=  96  agreement=92.7%  both correct=88.5%  (A=93.8%, B=91.7%)
  by congruency of the instruction with its own target:
    congruent    n= 491  agreement=58.0%
    incongruent  n= 189  agreement=77.2%
--- at least one bin (0.000325)
  n=259  (81 scenes gave no decidable prediction and pick no side)
  instructions moving toward their own target: 65.8%
  both instructions correct in the same scene: 44.0%
  by configuration:
    opposite         n= 117  agreement=47.0%  both correct=5.1%  (A=25.6%, B=68.4%)
    same_side_left   n=  56  agreement=61.6%  both correct=53.6%  (A=58.9%, B=

## 8. Controls

### 8a. Lexical prior

The same instruction contrast, run against a scene the instruction does not
describe. A difference that remains at comparable size is attributable to the
words rather than to the scene, and the baseline contrast cannot then be read as
grounding.

The comparison image is assigned by a derangement, so no scene receives its own
frame. The replacement is a real scene rather than a grey or noise field as an
input far outside the training distribution can collapse the action to a
constant, which is indistinguishable from an absent lexical prior.

In [10]:
for source in sorted(usable['scene_source'].unique()):
    frame = usable[usable['scene_source'] == source]
    result = lexical_check(frame)
    print(f'--- {source}')
    for label in ('baseline', 'swapped_scene'):
        entry = result[label]
        if not entry.get('n'):
            print(f'  {label}: not present')
            continue
        print(f"  {label:14} n={entry['n']:4}  mean |difference|={entry['mean_abs']:.6f}  "
              f"median={entry['median']:+.6f}  p={entry['p_value']:.3g}")
    if np.isfinite(result['ratio']):
        print(f"  ratio swapped/baseline = {result['ratio']:.2f}  "
              "(near 1 means the contrast is reproduced without the scene, "
              "near 0 means it depends on the scene)")

--- constructed
  baseline       n= 340  mean |difference|=0.003934  median=-0.000006  p=0.485
  swapped_scene  n= 340  mean |difference|=0.004519  median=-0.000000  p=0.559
  ratio swapped/baseline = 1.15  (near 1 means the contrast is reproduced without the scene, near 0 means it depends on the scene)


### 8b. The term's marginal effect

Each instruction expressed as a deviation from the prediction on the identical
image with the spatial term removed. Measuring against a within-scene reference
removes whatever the scene contributes on its own, which the raw difference
between the two instructions cannot.

If the term carries directional information, the two variants deviate from that
reference in opposite directions and the product of the deviations is negative.

In [11]:
for source in sorted(usable['scene_source'].unique()):
    frame = usable[usable['scene_source'] == source]
    result = term_effect(frame)
    if not result.get('n'):
        print(f'--- {source}: no scenes with both a baseline pair and a neutral '
              'reference')
        continue
    print(f"--- {source}  n={result['n']}")
    print(f"  deviations in opposite directions: {result['opposed_rate']:.1%}")
    print(f"  both deviations exactly zero:      {result['both_zero_rate']:.1%}")
    for label in ('deviation_a', 'deviation_b'):
        entry = result[label]
        print(f"  {label:12} median={entry['median']:+.6f}  p={entry['p_value']:.3g}")
    sep = result['separation']
    print(f"  separation between them: median={sep['median']:+.6f}  "
          f"p={sep['p_value']:.3g}  effect={sep['rank_biserial']:+.2f}")

--- constructed  n=340
  deviations in opposite directions: 39.1%
  both deviations exactly zero:      0.0%
  deviation_a  median=-0.000017  p=0.129
  deviation_b  median=+0.000000  p=0.825
  separation between them: median=-0.000006  p=0.485  effect=-0.04


## 9. Equivalence testing for nulls

A comparison that shows no effect leaves open whether the effect is smaller than
the bound of interest or merely unproven. A non-significant p value does not
separate those cases. Each null is therefore tested for equivalence to zero
within one action bin width, a bound set by the instrument. A difference smaller
than one bin cannot change the action the model would execute.

An equivalent result supports the claim that the manipulation does not move the
model. A result that is neither significantly different nor significantly
equivalent means the sample cannot resolve the question.

The two tests address different claims, so both can reject. A small, consistent
difference in a large sample can be detectable while remaining smaller than one
action bin: present in the readout, but too small to change the executable
action. That case is reported as `detectable but sub-bin` rather than as an
effect or a null.

In [12]:
rows = []
for source in sorted(usable['scene_source'].unique()):
    frame = usable[usable['scene_source'] == source]
    for condition in ('baseline', 'swapped_scene'):
        diffs = paired_difference(frame, condition=condition)['diff']
        if diffs.empty:
            continue
        difference = wilcoxon_paired(diffs)
        equivalence = tost_equivalence(diffs, BIN_WIDTH)
        significant = difference['p_value'] < 0.05
        if significant and equivalence['equivalent']:
            verdict = 'detectable but sub-bin'
        elif significant:
            verdict = 'effect present'
        elif equivalence['equivalent']:
            verdict = 'equivalent to zero'
        else:
            verdict = 'inconclusive'
        rows.append({
            'scene_source': source, 'condition': condition, 'n': difference['n'],
            'median': difference['median'], 'p_difference': difference['p_value'],
            'p_equivalence': equivalence['p_equivalence'], 'verdict': verdict,
        })

if rows:
    print(pd.DataFrame(rows).to_string(index=False))
    print(f'\nequivalence bound: +/- {BIN_WIDTH:.7f} (one action bin)')

scene_source     condition   n        median  p_difference  p_equivalence      verdict
 constructed      baseline 340 -5.640490e-06      0.484839       0.684202 inconclusive
 constructed swapped_scene 340 -7.217750e-09      0.558941       0.701005 inconclusive

equivalence bound: +/- 0.0003250 (one action bin)


## 10. Manipulation check

A composited object is only a usable stimulus if the model treats it as an
object. Under the term-free instruction, which names the object without locating
it, the predicted action should sometimes point at the pasted instance rather
than the original. A rate at or near zero means the paste is ignored, and the
constructed scenes cannot support the language comparison. That outcome would
also indicate that a box cutout is insufficient and that an outline mask is
required.

The expectation is one-sided. A model free to choose between two instances need
not split evenly. The check is that the duplicate is selected a non-trivial
fraction of the time, not that the split is balanced.

In [13]:
# The frozen set, so the check describes the scenes the experiments ran on.
scenes = evaluation_scenes(CONSTRUCTED_DIR)
if scenes:
    neutral = usable[(usable['scene_source'] == 'constructed')
                     & (usable['condition'] == 'neutral')].copy()
    neutral['construct_id'] = neutral['scene_id'].astype(str)
    result = manipulation_rate(neutral, scenes, value_col='c0')
    if result['n']:
        print(f"informative scenes (instances in opposite directions): {result['n']}")
        print(f"action toward the pasted instance: {result['toward_pasted']} "
              f"({result['rate']:.1%})")
        for name, entry in sorted(result['by_configuration'].items()):
            print(f"  {name:16} n={entry['n']:4}  toward pasted={entry['rate']:.1%}")
    else:
        print('no informative neutral predictions on constructed scenes yet')
else:
    print('no frozen evaluation set, screen and freeze in Notebook 04 first')

informative scenes (instances in opposite directions): 166
action toward the pasted instance: 83 (50.0%)
  opposite         n= 133  toward pasted=54.1%
  same_side_left   n=  12  toward pasted=25.0%
  same_side_right  n=  21  toward pasted=38.1%


## 11. Summary

Collects the numbers the write-up depends on into one place, so the reported
result and the conditions under which it holds cannot drift apart.

In [14]:
summary = {}

entry = mirror_built_same_side.get('neutral', {})
if entry.get('n'):
    summary['mirror flip rate (same side, term free)'] = f"{entry['flip_rate']:.1%}"

if not built.empty:
    congruence = congruence_test(built, lateral_sign=IMAGE_X_TO_LATERAL_SIGN)
    summary['paired difference matches geometry'] = f"{congruence['agreement']:.1%}"
    decisive = same_side_test(built, min_magnitude=BIN_WIDTH)
    paired = decisive['contrast_paired']
    summary['same-side minus opposite, paired'] = (
        f"{paired['difference']:+.1%} (p={paired['p_value']:.3g}, "
        f"{paired['n_pairs']} frames)")
    summary['same-side minus opposite, unpaired'] = (
        f"{decisive['contrast']['difference']:+.1%} "
        f"(p={decisive['contrast']['p_value']:.3g})")
    absolute = absolute_congruence(built, lateral_sign=IMAGE_X_TO_LATERAL_SIGN,
                                   min_magnitude=BIN_WIDTH)
    for name in ('same_side_left', 'same_side_right'):
        entry = absolute.get('by_configuration', {}).get(name)
        if entry and entry.get('n'):
            summary[f'moves toward own target ({name})'] = (
                f"{entry['agreement']:.1%} (n={entry['n']})")
    incongruent = absolute.get('by_congruency', {}).get('incongruent', {})
    if incongruent.get('n'):
        summary['moves toward own target (incongruent term)'] = (
            f"{incongruent['agreement']:.1%} (n={incongruent['n']})")
    summary['sign convention (IMAGE_X_TO_LATERAL_SIGN)'] = f'{IMAGE_X_TO_LATERAL_SIGN:+d}'

lex = lexical_check(usable)
if np.isfinite(lex['ratio']):
    summary['lexical ratio, swapped over baseline'] = f"{lex['ratio']:.2f}"

continuous_diffs = paired_difference(usable)['diff']
argmax_diffs = paired_difference(usable, continuous=False)['diff']
if len(continuous_diffs) and len(argmax_diffs):
    summary['exact zeros, argmax readout'] = (
        f"{resolution_report(argmax_diffs, BIN_WIDTH)['frac_exact_zero']:.1%}")
    summary['exact zeros, continuous readout'] = (
        f"{resolution_report(continuous_diffs, BIN_WIDTH)['frac_exact_zero']:.1%}")

width = max(len(k) for k in summary) if summary else 0
for key, value in summary.items():
    print(f'{key:<{width}}  {value}')

mirror flip rate (same side, term free)     57.7%
paired difference matches geometry          46.8%
same-side minus opposite, paired            +4.1% (p=0.549, 74 frames)
same-side minus opposite, unpaired          +5.7% (p=0.199)
moves toward own target (same_side_left)    61.6% (n=56)
moves toward own target (same_side_right)   94.2% (n=86)
moves toward own target (incongruent term)  83.8% (n=142)
sign convention (IMAGE_X_TO_LATERAL_SIGN)   -1
lexical ratio, swapped over baseline        1.15
exact zeros, argmax readout                 32.6%
exact zeros, continuous readout             0.0%
